# Retail daily-sales forecasting

Reproducible **synthetic demo** of the original academic forecast workflow. To reproduce original figures, provide your own lawfully accessible aggregated daily-sales CSV and update `INPUT_PATH`. Original course data are not included.

In [ ]:
from pathlib import Path
import sys

# Run this notebook from the project root.
INPUT_PATH = Path("data/sample_daily_sales.csv")
OUTPUT_DIR = Path("results")
sys.path.insert(0, str(Path("src").resolve()))
from forecast import load_data, build_features, forecast, FEATURES

sales = load_data(INPUT_PATH)
print(f"{len(sales)} days, {sales.Date.min().date()} to {sales.Date.max().date()}")
sales.head()

## 1. Lag features

All predictors use the date or past observed sales—not the target day’s sales. The first seven days have incomplete lags.

In [ ]:
model_data = build_features(sales)
model_data[["Date", "DailySales"] + FEATURES].head()

## 2. Chronological validation

The last 14 dates are held out. This tests rolling **one-day-ahead** predictions with true historical lags, not a 14-day recursive forecast.

In [ ]:
test, rf_mae, baseline_mae, next_date, prediction, n_train = forecast(sales, test_days=14)
print(f"Train rows: {n_train}; test rows: {len(test)}")
print(f"Random Forest MAE: ${rf_mae:,.2f}")
print(f"Previous-week baseline MAE: ${baseline_mae:,.2f}")
print(f"One-day-ahead estimate for {next_date.date()}: ${prediction:,.2f}")

## 3. Compare actual and predicted sales

The dashed line is the previous-week baseline. The model has nontrivial uncertainty and should not be used to make precise inventory decisions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test["Date"], test["DailySales"], marker="o", label="Actual")
ax.plot(test["Date"], test["RandomForest"], marker="o", label="Random Forest")
ax.plot(test["Date"], test["PreviousWeekBaseline"], linestyle="--", label="Previous-week baseline")
ax.set(xlabel="Date", ylabel="Daily sales ($)", title="Rolling one-day-ahead forecast")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend();ax.grid(alpha=0.2);fig.autofmt_xdate();fig.tight_layout()
plt.show()

## Limitations

Synthetic demonstration values **cannot reproduce the original case-study statistics**. Original test set was only 14 days long; model errors were substantial. The business dataset lacks detailed inventory and promotional variables. The next-day estimate has no observed ground truth.